<img src="https://devra.ai/analyst/notebook/1001/image.jpg" style="width: 100%; height: auto;" />

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">  <div style="font-size:150%; color:#FEE100"><b>Win-Go Color Prediction Analysis</b></div>  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div></div>

## Table of Contents

- [Introduction](#Introduction)
- [Imports and Setup](#Imports-and-Setup)
- [Data Loading and Exploration](#Data-Loading-and-Exploration)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis](#Exploratory-Data-Analysis)
- [Predictive Modeling](#Predictive-Modeling)
- [Model Evaluation and Visualization](#Model-Evaluation-and-Visualization)
- [Permutation Importance](#Permutation-Importance)
- [Conclusions and Future Work](#Conclusions-and-Future-Work)

If you find this notebook useful, please upvote it.

## Introduction

This analysis investigates the Win-Go Color Prediction dataset. The dataset comprises four columns: Period, Price, Number, and Color. One might say that the interplay between these numerical indicators and a categorical color outcome hides an interesting story. Our goal is to explore the data, visualize underlying trends, and even build a classification model to predict the color outcome based on the numeric features. Additionally, we will provide diagnostic visualizations such as confusion matrices and permutation importance (if the required module is available) to help understand the model's behavior.

In [ ]:
# Import necessary libraries and set up the environment
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Use Agg backend for matplotlib
import matplotlib.pyplot as plt
plt.switch_backend('Agg')  # In case only plt is imported

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Ensure inline plotting in Kaggle notebooks
%matplotlib inline

sns.set(style='whitegrid', palette='muted')

## Data Loading and Exploration

In [ ]:
# Load the dataset from the CSV file
file_path = '/kaggle/input/wingo-color-prediction/Win-Go.csv.csv'
df = pd.read_csv(file_path, encoding='ascii', delimiter=',')

# Display the first few rows
df.head()

In [ ]:
# Basic data exploration: structure, summary statistics, and data types
print('DataFrame Info:')
df.info()

print('\nSummary Statistics:')
df.describe(include='all')

# Check for missing values
print('\nMissing Values:')
print(df.isnull().sum())

## Data Cleaning and Preprocessing

The dataset appears to be relatively clean as it comes from a curated source. However, it is always prudent to inspect the data closely. In this notebook, we check for missing values and review types. Note that in other datasets, methods such as imputation or type conversion (for dates, for example) might become necessary.

In [ ]:
# In this dataset, there are no explicit date columns, and categorical column is 'Color'.
# For a real-world problem, we might need to convert dates or handle missing values here.

print('Unique values in Color:', df['Color'].unique())

# Sometimes, the numeric features might need scaling or outlier treatment. Here, we assume the dataset is ready for analysis.

## Exploratory Data Analysis

In this section, we explore the data using various visualization techniques to get a feel for the distribution of our features and the target variable. We utilize several plotting methods to uncover any interesting trends.

In [ ]:
# Count Plot (Pie chart equivalent) for Color distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Color', data=df, palette='pastel')
plt.title('Distribution of Colors')
plt.xlabel('Color')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

# Histograms for numerical features: Period, Price, and Number
num_features = ['Period', 'Price', 'Number']
for feature in num_features:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[feature], kde=True, color='skyblue')
    plt.title(f'Histogram of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

# Pair Plot to examine relationships between numeric features
sns.pairplot(df[num_features])
plt.suptitle('Pair Plot of Numeric Features', y=1.02)
plt.show()

# Note: A correlation heatmap is typically generated if there are 4 or more numeric columns. 
if len(num_features) >= 4:
    numeric_df = df[num_features]
    plt.figure(figsize=(8, 6))
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Heatmap')
    plt.show()
else:
    print('Skipping correlation heatmap as there are less than 4 numeric features.')

## Predictive Modeling

We now transition into building a predictive model. The task is to predict the 'Color' category using the numeric features: Period, Price, and Number. Given the modest size and the clear distinction in variables, a Random Forest classifier should be appropriate and robust.

In [ ]:
# Prepare feature variables and target variable
X = df[['Period', 'Price', 'Number']]
y = df['Color']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Build and train the Random Forest Classifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Model Accuracy: {accuracy:.3f}')

## Model Evaluation and Visualization

A confusion matrix will be used to visually assess the performance of our classifier. This step allows us to identify patterns in misclassification.

In [ ]:
# Compute the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot the confusion matrix using a heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## Permutation Importance

Permutation importance can provide insights into which features are most influential in the classifier decisions. Note that the `eli5` package is required for this analysis. If this package is not installed (or the environment does not support it), this section will note the error encountered.

The following cell attempts to compute and visualize permutation importance. If you encounter an error about the module not being found, consider installing `eli5` in your environment.

In [ ]:
try:
    import eli5
    from eli5.sklearn import PermutationImportance

    # Compute permutation importance on the test set
    perm = PermutationImportance(model, random_state=42).fit(X_test, y_test)

    # Format the feature importances into a dataframe
    feature_importances = eli5.format_as_dataframe(eli5.explain_weights(perm, feature_names=X_test.columns.tolist()))

    # Plot the permutation importances using a horizontal bar plot
    plt.figure(figsize=(8, 4))
    plt.barh(feature_importances['feature'], feature_importances['weight'], color='teal')
    plt.xlabel('Permutation Importance')
    plt.ylabel('Features')
    plt.title('Feature Importance based on Permutation')
    plt.tight_layout()
    plt.show()

except ModuleNotFoundError as e:
    print('eli5 module not found. Please install it to compute permutation importance (e.g., pip install eli5).')

## Conclusions and Future Work

This notebook presented a comprehensive approach starting from data loading and exploration, moving through preprocessing and visualization, and finally developing a classification model to predict the color based on numeric features. A Random Forest classifier provided a reasonably good predictive performance, and diagnostic plots such as the confusion matrix and permutation importance help in understanding model behavior.

Merits of the approach:
- A diverse range of visualization techniques was applied to reveal underlying trends.
- Predictive modeling allowed us to validate the relationship between features and the target variable.
- The notebook includes practical considerations for troubleshooting module errors, as shown in the permutation importance section.

Future analysis might include:
- Experimenting with additional models or hyperparameter tuning for improved prediction accuracy.
- Investigating potential interactions between features or incorporating feature engineering techniques if more data were available.
- Expanding the permutation importance analysis to capture more complex relationships between variables.

If you found this analysis insightful, please upvote the notebook.